In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [3]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import roc_curve, roc_auc_score

from src.config import CONFIG
from src.preprocess import preprocess_data, split_data, apply_smote
from src.train import train_xgboost, compute_scale_pos_weight
from src.evaluate import evaluate

In [7]:
df = pd.read_csv("../data/creditcard.csv")

df_clean = preprocess_data(df)

X_train, X_test, y_train, y_test = split_data(
    df_clean,
    CONFIG["target_column"],
    CONFIG["test_size"],
    CONFIG["random_state"]
)

In [8]:
X_smote, y_smote = apply_smote(
    X_train,
    y_train,
    CONFIG["random_state"]
)

In [16]:
weight = compute_scale_pos_weight(y_smote)

params = CONFIG["xgb_params"].copy()
params["scale_pos_weight"] = weight

model = train_xgboost(
    X_smote,
    y_smote,
    X_test,
    y_test,
    params
)

In [22]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

y_prob = model.predict_proba(X_test)[:,1]

thresholds = [0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5,0.9]

results = []

for t in thresholds:
    
    y_pred = (y_prob >= t).astype(int)

    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    results.append([t, precision, recall, f1])

df_results = pd.DataFrame(
    results,
    columns=["threshold", "precision", "recall", "f1"]
)

df_results

,threshold,precision,recall,f1
0,0.01,0.294776,0.831579,0.435262
1,0.02,0.386139,0.821053,0.525253
2,0.05,0.516556,0.821053,0.634146
3,0.10,0.617886,0.800000,0.697248
4,0.20,0.723810,0.800000,0.760000
5,0.30,0.783505,0.800000,0.791667
6,0.50,0.844444,0.800000,0.821622
7,0.90,0.937500,0.789474,0.857143
